# 35. One golden record from three systems

**Track:** pyspark | **Difficulty:** hard | **Tags:** master data, survivorship, deduplication, window functions, first ignore nulls

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("035-golden-customer-record")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "customer_records": """
UEFSMRUEFdAFFYYDTBVaFQASAADoAgQJAAkBABAJBwQAIg0IABYNCAASDQgAJw0IABcNCAAMDQgADg0IACkNCAADDQgAAg0IACAN
CAAjDQgAHw0IAAENCAAYDQgAJA0IBB0ACQEAFAkHBAATDQgADw0IACYNCAAFDQgAGQ0IABsNCAAcDQgAEQ0IAB4NCAAHDQgAKw0I
ACwNCAAaDQgACw0IACoNCAAoDQgABg0IAAgNCAAtDQgACg0IACUNCAAVDQgADQ0IPAQAAAAAAAAAIQAAAAAAAAAVABXSARXYASwV
9AEVEBUGFQYcGAgtAAAAAAAAABgIAQAAAAAAAAAWACgILQAAAAAAAAAYCAEAAAAAAAAAEREAAABp8GgDAAAA9AEBBiFAIAxEYRxI
IigLwwCN4TxQJE1UZVjEhRFQpjUKVUUbBzHU03kCET3fAoZi2I0DFgzkJ4llIpVHGUKkeXpoxFkpd4SLOoDcRD2NApVhqUjhqZ3C
ymqKhiETOCKTCAAAAAAVBBU6FT5MFQYVABIAAB1wAwAAAGNybQcAAABiaWxsaW5nBwAAAHN1cHBvcnQVABVSFVYsFfQBFRAVBhUG
HDYAKAdzdXBwb3J0GAdiaWxsaW5nEREAAAApoAMAAAD0AQECISSSBoBRCRkoQpWpioQoCJAmgFQqBVJRBQpUZlUYCgUAFQQV0h4V
6ghMFZYBFQASAACpD1wTAAAAYzkuY3JtMEBleGFtcGxlLmNvbRgBFyQxNi5iaWxsaW5nRhwAJDM0LnN1cHBvcnRGHAAEMjNmOAAA
MVY4AAAXAXAINDFADXAALh2LABQBGwQzNEKjAAW6ADMBFwAxLroACS8AMm4YADKyAAV6BDkuYs0ABDM1Us0ABWcAMgV/Rk8AADFq
MwAAMAGaMjMAKWwAOVI0AAkcADVSnwAFgwQzOEJQAAXSajMAADEFMzI1AQVnADI1mzIcAAVnBDI3QmcACRgFTEbPADk2MkwABWgA
MVZSAQkcADdmBwEEMzBSBwEFOAQzMmYcAAAxVt4BRUBG1wAlPgg0M0BNEVqKAjLWACUKBDM2QiIBBWIUNDRAY3JtLkcABWJqsAAE
MjZSzAAFZgQxMQFmMn4ACWYAMj5mAAUvADRFbjIvACUvADJWfwAFmhV+MjcABRsUNDIub2xkPmoABYFGCAIFgQAyhVtGSgAAOGrj
AGW8MlMCBX0AOGYaAgAxVnEDBTcENDVOywEFtAQ0Mz7LACUdADNW9wQF0GlvMtAACRglaTK1AAVMBDEyUrYABZpWGwAFTwAxRlkE
BU8AMlY0AAkcVrcABRwEMzBmhwAAM2aLAQAyRnYFCZ9FhjLuAAXSNaMyIQEFggQxMRFnShwAFZ5GHAAAMlUqRhwABDM0UtYABXAA
M2qEBgAyFagywwBFFF55ARWHMjMABWsEMjAR2zIcACkqRkIBETTN6zI0ACVGBDQyTgcHFQAVugEVwAEsFfQBFRAVBhUGHDZeKBdj
OS5zdXBwb3J0MEBleGFtcGxlLmNvbRgXYzEuYmlsbGluZzBAZXhhbXBsZS5jb20REQAAAF3wXBEAAAAhR++Lq0/b9YnZp2ofNteT
AwcVgIBgQCgYDoiEYsFoOB6QiGRCqVgumIxmw+l4PqCQaEQqmU6olGrFarlesJhsRqvZbricbsfr+X7AoHBILBqPyKQSAAAAABUE
FYwPFbQLTBWKARUAEgAAxgc0CgAAADkyODQ2MDIzODQJDhwwNDA5NTUzMQUOIDEzMzgxNTQxMwUOIDYyMTYwOTYwMAUOADAFAQg0
MTIFDiA4OTU2MTkyNTUFDiAwODQ2MTEwNjYFDiA2MDc2NjU0NDgFDiA3NjU1MjMxMjkFDhwyMDUxNDUyNwkqHDAwNjk3Nzk5CX4g
NjY0OTI2MjI3BSocNTkwOTc1MTIJjBwyODY0ODA0NQmMHDQzMjA3NDEyDX4ENTQBbgA5CWIYMTUzNDA3Mg22GDk1ODQ4NjEN7hQ1
NDg4NjgRKiAwNjg5NjUyMDQFcBw3NDE1MzYxMw1UGDYwMDYyOTEJDiA5OTc1MjU4MzgFKhwzMjQ4MjUzMAkcHDc3MTMzMDIwCbYY
MTA5NDE1Ng3SGDM3OTc1OTUNOCAwNjgyMTIzNTYFRiVPCDA0MxHgFDgxODc1MQlGGDU2MzMyNDUtQhg4NjY2NDczDVQcNTA5Njc3
NzkJthwwMTIzMjc2NS2WGDMyODk0NTkJYhw1NzgxNTg0Mg1UFDcwMjk4NQ3SDX4AMil6DQ4AMQkOBDc3IUMENDQJYhwxNTgyMjAx
MQkOHDk2NDAyOTk4CSoNOAAzCQ4cMjA1NjY1NTUJKhw1MDQwMTExMik0HDgzMDM1MzE1CYwEODABigQ3MymyBDg0AScAMC16HDM4
MDQyNDExCRwcNTgyMTc3NjYJRhwyMzQwMzc2NAngHDkzNTU3NTk4CQ4YODU0ODI5OA3gHDczMzk1MzcxCTgANCH0DDkzODUpegg4
NDUhUQmMHDEwNTY3NTM4CQ4YMzYyMzg4NE3YHDEwNTEyODY5CRwYNjQ3NzM3NS2yHDA2MzIxNTIyCe4cNjA5MDA0OTQpzhg3NDY4
MTIxTbwcOTE4MzkwNDAJ0gQzMgE6BDg5CYwYNTU2MTUyOU2uGDUyMDMzMjAN0lgxNjkwNDIxMDUKAAAAOTAwMDAwMDQyMhUAFawB
FbIBLBX0ARUQFQYVBhw2aigKOTk5NzUyNTgzOBgKOTAwMDAwMDQxMRERAAAAVvBVEQAAACFJ27l0W9tlPKiY+nKeracCBxOAgGBA
KBgOiIRiwWg4HpCIZEKpWC6YjGbD6Xg+oJBoRCqZTqiUasVquV6wmGxGq9luuJxux+v5fsCgcEgEAAAVBBWqARWqAUwVEBUAEgAA
VVwHAAAAS29sa2F0YQkAAABIeWRlcmFiYWQBGGBDaGVubmFpBAAAAFB1bmUGAAAASmFpcHVyAQpsTXVtYmFpBQAAAERlbGhpCQAA
AEJlbmdhbHVydRUAFXoVgAEsFfQBFRAVBhUGHDY+KARQdW5lGAlCZW5nYWx1cnUREQAAAD3wPBMAAAAD9RABHX7N791Pv/v3ydzn
dH4DAxmIJrZMkhg+Z7N2QDSLQFzF3fLRYm3akV8KiWqVv4MI8Uc4AQAVBBWgDxXuDUwV9AEVABIAANAH8EwADUXheT0GAABuOjS1
OgYAAFFQ7msrBgAAHM58OzkGAADAi7oqMgYAAJXXtUQ4BgAAmdYrDDQGAADVb2hxNQYAAJELzQcuBgAAyMUYEAEYDJ8zG4cBOCzP
u3h9LQYAADdPKvEBEAxJBqjXAQgs3KlDxzsGAAC2iqQHAXAM9I4sgQFgLN52J6E2BgAA40UiKAEgLDa/0XM3BgAAf6vIBgFIDCUa
TcoBQAyOrMHhAQgMKPRExAG4DDGYMtYBOCyeu8FxMAYAAHrf/xsBIAztnmP1AVgMdeLVkwG4DKena+sBsCwid7iUMQYAAD0jFLgB
OAztTGTnAWgMjFUmjwEgDCg5wWgBOAxrZH06ASAsgaP9Lz4GAADyQr41AbAsOcZU0T0GAAA6hflZASgs0JkWHS8GAABHtaNmAQgs
LDiwnCoGAAArql6KAcgMBGJdnAGYDAiB96sBUAzc/3UtASgM6EBvugFIDEm5jeYBgAy35qHnATAMw0ndggFYDGl3CtMBEAyIiJm0
AbAssE723jQGAAB9g7FbATAMnGrZTgEQDGjatX8BWAyu8nVvARgMwF0SvSEADAtmh/IBQAwtTn8TARgMs1XyqAEIDMdpGzABYCzo
xYLmLQYAAA2bE+ABoAzoX7fIAWgMa71aqQFQDGjluBEBICz5L+rfMwYAAOhz/EQBuAzUOZEAARgM+qhg/wFICLpbZCUoDAcxNrYB
IAxZsyvwISgMjnBr2QHgDKVNTwUBEAxILNJKAZAM07dtsQFoDAiYvbtBaCxz2+PsNAYAAAXRgu4BuAzp2d+sASAMjzIYiQFoDMJt
KgABkCyRrEqQLAYAABdo6rMBEAyTX1IMAbgMGti5wgEIDM/EM9QBCAxuRbCzATgMqTc50QEwDFasLyZBAAzF9RTFIQAMBtcVzAF4
DE6jj/wBMAyQmwMYARAM3qYzmAEQDOytuTEBMAyjpoNzAYgMd/B7JgFILPXVJPkuBgAAef7EmgEYDLGfW3JBMAwYuU9HAVgMM8jL
fwFIDEqFw/8ByAzndKTmAVAMEuSllCEoDDMEt0IBCAxcs7SeAaAMjGnyngHIDIQh5h4BUAwDu7d7AVAsKWKO3DAGAACYGCshAUhM
lrF9qi8GAADiYhjYPAYAAKeDFEshkAyavNI9Aag4h31wsS0GAACoFzXbOAYAFQAV8gEV+AEsFfQBFRAVBhUGHBgIAIGj/S8+BgAY
CAAsOLCcKgYAFgAoCACBo/0vPgYAGAgALDiwnCoGABERAAAAefB4AwAAAPQBAQchgIBgQCgYDoiEYsFoOB6QiGRCqVgumIxmw+l4
PqCQaEQqmU6olGrFarlesJhsRqvZbricbsfr+X7AoHBILBqPyKRyyWw6n9CodEqtWq/YrHbL7Xq/4LB4TC6bz+i0es1uu9/wuHxO
r9vv+DwAAAAAABUEGXw1ABgGc2NoZW1hFQwAFQQlAhgLY3VzdG9tZXJfaWQAFQwlAhgGc291cmNlJQBMHAAAABUMJQIYBWVtYWls
JQBMHAAAABUMJQIYBXBob25lJQBMHAAAABUMJQIYBGNpdHklAEwcAAAAFQQlAhgKdXBkYXRlZF9hdCUUTIwSHCwAAAAAABb0ARkc
GWwmABwVBBk1AAYQGRgLY3VzdG9tZXJfaWQVAhb0ARbGCBaCBiauAyYIHBgILQAAAAAAAAAYCAEAAAAAAAAAFgAoCC0AAAAAAAAA
GAgBAAAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgD0AQAAACYAHBUMGTUABhAZGAZzb3VyY2UVAhb0ARb8ARaEAibkBiaK
Bhw2ACgHc3VwcG9ydBgHYmlsbGluZxERABksFQQVABUCABUAFRAVAgA8FsQKGQYZJgD0AQAAACYAHBUMGTUABhAZGAVlbWFpbBUC
FvQBFsYhFuQLJpoRJo4IHDZeKBdjOS5zdXBwb3J0MEBleGFtcGxlLmNvbRgXYzEuYmlsbGluZzBAZXhhbXBsZS5jb20REQAZLBUE
FQAVAgAVABUQFQIAPBb6GRkGGSZelgEAAAAmABwVDBk1AAYQGRgFcGhvbmUVAhb0ARa+ERbsDSbIHybyExw2aigKOTk5NzUyNTgz
OBgKOTAwMDAwMDQxMRERABksFQQVABUCABUAFRAVAgA8FuQKGQYZJmqKAQAAACYAHBUMGTUABhAZGARjaXR5FQIW9AEWmAMWngMm
qCMm3iEcNj4oBFB1bmUYCUJlbmdhbHVydRERABksFQQVABUCABUAFRAVAgA8FuIJGQYZJj62AQAAACYAHBUEGTUABhAZGAp1cGRh
dGVkX2F0FQIW9AEWuBIWjBEmjDMm/CQcGAgAgaP9Lz4GABgIACw4sJwqBgAWACgIAIGj/S8+BgAYCAAsOLCcKgYAEREAGSwVBBUA
FQIAFQAVEBUCADwpBhkmAPQBAAAAFpZTFvQBJggWgDYAGRwYDEFSUk9XOnNjaGVtYRj4Ay8vLy8vM0FCQUFBUUFBQUFBQUFLQUF3
QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFZQUFBQUVBUUFBeEFBQUFKZ0FBQUJzQUFBQVFB
QUFBQVFBQUFBay8vLy9BQUFCQ2hBQUFBQWtBQUFBQkFBQUFBQUFBQUFLQUFBQWRYQmtZWFJsWkY5aGRBQUFBQUFHQUFnQUJnQUdB
QUFBQUFBQ0FGei8vLzhBQUFFRkVBQUFBQmdBQUFBRUFBQUFBQUFBQUFRQUFBQmphWFI1QUFBQUFJai8vLytFLy8vL0FBQUJCUkFB
QUFBWUFBQUFCQUFBQUFBQUFBQUZBQUFBY0dodmJtVUFBQUN3Ly8vL3JQLy8vd0FBQVFVUUFBQUFHQUFBQUFRQUFBQUFBQUFBQlFB
QUFHVnRZV2xzQUFBQTJQLy8vOVQvLy84QUFBRUZFQUFBQUJ3QUFBQUVBQUFBQUFBQUFBWUFBQUJ6YjNWeVkyVUFBQVFBQkFBRUFB
QUFFQUFVQUFnQUJnQUhBQXdBQUFBUUFCQUFBQUFBQUFFQ0VBQUFBQ1FBQUFBRUFBQUFBQUFBQUFzQUFBQmpkWE4wYjIxbGNsOXBa
QUFJQUF3QUNBQUhBQWdBQUFBQUFBQUJRQUFBQUE9PQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGWwcAAAcAAAc
AAAcAAAcAAAcAAAASAUAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "035-golden-customer-record"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

customer_records = spark.read.parquet((_data_dir / "customer_records.parquet").resolve().as_posix())
customer_records.createOrReplaceTempView("customer_records")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'city': 'cf423995b1b3415c7fdb75e69ee6e75b9610bffabd422ba298b5b6a6e30774a3',
                         'customer_id': 'dbed8193ea95bfaf44f638adfc915b4fba9e728a2e4716ad25fbe3f757600d9d',
                         'email': '906e8ce9bff645ea6a88bfec31546a77bf7d58139a30dd27c7e85b7640620a18',
                         'phone': '9d2f061aecea60a152d0298a0f4b5721b0d16d7550ed4e2a5fb0bf9e7e35bb20'},
 'columns': [['city', 'string'],
             ['customer_id', 'bigint'],
             ['email', 'string'],
             ['phone', 'string']],
 'fingerprint': '9ee5418cc2e4a341957ba2f89086fee8dce184226461c2d5c1c20d9667d13fc5',
 'order_matters': False,
 'row_count': 45}

check = make_check(EXPECTED)

## The situation

Three systems each keep their own copy of a customer: the CRM, billing and the
support desk. Marketing wants one row per customer with the best email, phone and
city the company knows.

The current job keeps the most trusted record of each customer and drops the rest.
The CRM record usually wins, and the CRM often has no phone number. Billing had one.
It was thrown away with the rest of the billing record.

## Input tables

### `customer_records`

One row per record of a customer in one system. A system can hold several records
for the same customer, saved at different times.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | The customer. Never NULL. |
| `source` | STRING | `crm`, `billing` or `support`. |
| `email` | STRING | NULL if this record has none. |
| `phone` | STRING | NULL if this record has none. |
| `city` | STRING | NULL if this record has none. |
| `updated_at` | TIMESTAMP_NTZ | When this record was last saved. |

A customer never has two records with the same `updated_at`.

## Requirement

Return one row per customer.

Rank a customer's records by trust:

1. By source: `crm` first, then `billing`, then `support`.
2. Within the same source, the record with the later `updated_at` first.

Each of `email`, `phone` and `city` is decided on its own: take the value from the
highest-ranked record in which that column is not NULL. The three values of one
customer can come from three different records.

If no record of the customer has a value for a column, that column is NULL.

## Expected output

One row per customer.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `email` | STRING | |
| `phone` | STRING | |
| `city` | STRING | |

In [ ]:
customer_records.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `customer_records`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)